# Outcome and Outlier Robustness: Singapore FY2023

This notebook repeats the matching and matched inference using operating revenue as an alternative size measure and after removing the largest 1% of firms separately within the listed and unlisted asset distributions. Each specification rebuilds the matched sample before applying the one-sided sign test, within-set permutation test (9,999 permutations), and matched-set bootstrap interval (5,000 resamples). Seed 607 is used throughout.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_FILE = (
    ROOT
    / "data"
    / "processed"
    / "orbis_singapore_fy2023_clean.parquet"
)

OUTPUT_FILE = (
    ROOT
    / "outputs"
    / "tables"
    / "robustness_outcomes.csv"
)

df = pd.read_parquet(DATA_FILE)

print("Data file:", DATA_FILE)
print("Original shape:", df.shape)

Data file: /Users/jiangminglan/ds607-singapore-firm-size/data/processed/orbis_singapore_fy2023_clean.parquet
Original shape: (48995, 34)


In [2]:
# Listing status
df["listing_status"] = df["listed"].map(
    {1: "Listed", 0: "Unlisted"}
)

# Firm age
df["incorporation_year"] = pd.to_numeric(
    df["incorporation_year"],
    errors="coerce"
)

df["firm_age"] = 2023 - df["incorporation_year"]
df.loc[df["firm_age"] < 0, "firm_age"] = np.nan

df["age_band"] = pd.cut(
    df["firm_age"],
    bins=[-1, 4, 9, 19, float("inf")],
    labels=["0–4", "5–9", "10–19", "20+"],
)

# Broad consolidation status: C or U
df["consolidated"] = (
    df["consolidation_code"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str[0]
)

df["consolidated"] = df["consolidated"].where(
    df["consolidated"].isin(["C", "U"])
)

# Exclude subsidiaries of sampled listed firms
df = df[df["listing_status"].notna()].copy()

n_subsidiaries = int(
    df["is_listed_subsidiary"]
    .fillna(False)
    .sum()
)

df = df[
    ~df["is_listed_subsidiary"].fillna(False)
].copy()

print("Excluded listed-firm subsidiaries:", n_subsidiaries)
print("Remaining firms:", len(df))
print(df["listing_status"].value_counts())

Excluded listed-firm subsidiaries: 2041
Remaining firms: 46954
listing_status
Unlisted    46458
Listed        496
Name: count, dtype: int64


In [3]:
df["revenue_th_usd"] = pd.to_numeric(
    df["revenue_th_usd"],
    errors="coerce"
)

df["revenue_usd"] = df["revenue_th_usd"] * 1_000

# Create the variable first, then calculate logs only for positive revenue
df["log_revenue"] = np.nan

positive_revenue = (
    df["revenue_usd"]
    .fillna(0)
    .gt(0)
)

df.loc[positive_revenue, "log_revenue"] = np.log(
    df.loc[
        positive_revenue,
        "revenue_usd",
    ].astype(float)
)

print(
    df.groupby("listing_status")
    ["log_revenue"]
    .count()
)

listing_status
Listed        491
Unlisted    36351
Name: log_revenue, dtype: int64


In [4]:
# 1. Main total-assets sample
assets_sample = df[
    df["assets_usd"].fillna(0).gt(0)
].copy()

# 2. Revenue robustness sample
revenue_sample = df[
    df["revenue_usd"].fillna(0).gt(0)
].copy()

# 3. Exclude the largest 1% separately within each listing group
top_1_thresholds = (
    assets_sample
    .groupby("listing_status")
    ["assets_usd"]
    .quantile(0.99)
)

threshold_for_each_row = (
    assets_sample["listing_status"]
    .map(top_1_thresholds)
    .astype(float)
)

assets_without_top_1 = assets_sample[
    assets_sample["assets_usd"]
    <= threshold_for_each_row
].copy()

print("Main assets sample:")
print(assets_sample["listing_status"].value_counts())

print("\nRevenue sample:")
print(revenue_sample["listing_status"].value_counts())

print("\nAssets sample after excluding the top 1%:")
print(assets_without_top_1["listing_status"].value_counts())

print("\n99th-percentile thresholds in USD:")
print(top_1_thresholds)

Main assets sample:
listing_status
Unlisted    45952
Listed        496
Name: count, dtype: int64

Revenue sample:
listing_status
Unlisted    36351
Listed        491
Name: count, dtype: int64

Assets sample after excluding the top 1%:
listing_status
Unlisted    45492
Listed        491
Name: count, dtype: int64

99th-percentile thresholds in USD:
listing_status
Listed      25483804932.780487
Unlisted      682074183.884236
Name: assets_usd, dtype: Float64


In [5]:
KEYS = ["nace2", "age_band", "consolidated"]
ID = "bvd_id"


def run_matching(
    data,
    outcome,
    max_controls=3,
    max_age_gap=None,
    seed=607,
):
    """
    Round-robin nearest-age matching.

    Each listed firm receives up to max_controls private controls:
    - same NACE 2-digit industry
    - same age band
    - same broad consolidation status
    - closest actual firm age
    - without replacement
    """

    rng = np.random.default_rng(seed)

    # Remove observations missing the outcome or matching variables
    base = data.dropna(
        subset=[outcome, "firm_age"] + KEYS
    ).copy()

    listed_all = base[
        base["listing_status"] == "Listed"
    ].copy()

    private_all = base[
        base["listing_status"] == "Unlisted"
    ].copy()

    # Divide private firms into matching strata
    private_groups = {
        key: group
        for key, group in private_all.groupby(
            KEYS,
            observed=True,
        )
    }

    matched_rows = []
    unmatched_rows = []
    set_number = 0

    # Match separately within each stratum
    for key, listed_group in listed_all.groupby(
        KEYS,
        observed=True,
    ):
        # Randomise listed-firm order within the stratum
        listed_group = listed_group.sample(
            frac=1,
            random_state=int(
                rng.integers(1_000_000)
            ),
        )

        private_group = private_groups.get(key)

        if private_group is None:
            for firm_id in listed_group[ID]:
                unmatched_rows.append(
                    {
                        ID: firm_id,
                        "reason": (
                            "no private firm in same stratum"
                        ),
                    }
                )
            continue

        available = private_group.assign(
            tiebreak=rng.random(
                len(private_group)
            )
        )

        picks = {
            idx: []
            for idx in listed_group.index
        }

        # Round-robin allocation:
        # everyone gets a first control before anyone gets a second
        for _ in range(max_controls):
            for idx, listed_row in listed_group.iterrows():

                if available.empty:
                    break

                age_gap = (
                    available["firm_age"]
                    - listed_row["firm_age"]
                ).abs()

                eligible = available.assign(
                    age_gap=age_gap
                )

                if max_age_gap is not None:
                    eligible = eligible[
                        eligible["age_gap"]
                        <= max_age_gap
                    ]

                if eligible.empty:
                    continue

                # Closest age, with random tie-breaking
                best_index = (
                    eligible
                    .sort_values(
                        ["age_gap", "tiebreak"]
                    )
                    .index[0]
                )

                picks[idx].append(
                    (
                        best_index,
                        age_gap[best_index],
                    )
                )

                # Without replacement
                available = available.drop(
                    best_index
                )

        # Store matched sets
        for idx, listed_row in listed_group.iterrows():

            if not picks[idx]:
                reason = (
                    "no unused private firm within age limit"
                    if max_age_gap is not None
                    else "stratum's private firms all used"
                )

                unmatched_rows.append(
                    {
                        ID: listed_row[ID],
                        "reason": reason,
                    }
                )
                continue

            set_number += 1

            matched_rows.append(
                {
                    "set_id": set_number,
                    "role": "listed",
                    ID: listed_row[ID],
                    "firm_age": listed_row["firm_age"],
                    "age_gap": 0.0,
                    "y": listed_row[outcome],
                }
            )

            for private_index, private_age_gap in picks[idx]:

                private_row = private_group.loc[
                    private_index
                ]

                matched_rows.append(
                    {
                        "set_id": set_number,
                        "role": "private",
                        ID: private_row[ID],
                        "firm_age": private_row[
                            "firm_age"
                        ],
                        "age_gap": private_age_gap,
                        "y": private_row[outcome],
                    }
                )

    matched = pd.DataFrame(matched_rows)

    unmatched = pd.DataFrame(
        unmatched_rows,
        columns=[ID, "reason"],
    )

    # Number of private controls in each matched set
    control_counts = (
        matched[
            matched["role"] == "private"
        ]
        .groupby("set_id")
        .size()
    )

    matched["n_controls"] = (
        matched["set_id"]
        .map(control_counts)
    )

    return (
        matched,
        unmatched,
        listed_all,
        private_all,
    )

In [6]:
# Main total-assets analysis
(
    matched_assets,
    unmatched_assets,
    eligible_listed_assets,
    eligible_private_assets,
) = run_matching(
    assets_sample,
    outcome="log_assets",
    max_controls=3,
    seed=607,
)


# Revenue robustness
(
    matched_revenue,
    unmatched_revenue,
    eligible_listed_revenue,
    eligible_private_revenue,
) = run_matching(
    revenue_sample,
    outcome="log_revenue",
    max_controls=3,
    seed=607,
)


# Excluding the largest 1%
(
    matched_trimmed,
    unmatched_trimmed,
    eligible_listed_trimmed,
    eligible_private_trimmed,
) = run_matching(
    assets_without_top_1,
    outcome="log_assets",
    max_controls=3,
    seed=607,
)

In [7]:
matching_check = pd.DataFrame(
    {
        "Specification": [
            "Main assets",
            "Operating revenue",
            "Assets excluding top 1%",
        ],
        "Eligible listed firms": [
            len(eligible_listed_assets),
            len(eligible_listed_revenue),
            len(eligible_listed_trimmed),
        ],
        "Eligible private firms": [
            len(eligible_private_assets),
            len(eligible_private_revenue),
            len(eligible_private_trimmed),
        ],
        "Matched sets": [
            matched_assets["set_id"].nunique(),
            matched_revenue["set_id"].nunique(),
            matched_trimmed["set_id"].nunique(),
        ],
        "Unmatched listed firms": [
            len(unmatched_assets),
            len(unmatched_revenue),
            len(unmatched_trimmed),
        ],
    }
)

matching_check

,Specification,Eligible listed firms,Eligible private firms,Matched sets,Unmatched listed firms
0,Main assets,496,45952,351,145
1,Operating revenue,491,36351,346,145
2,Assets excluding top 1%,491,45492,348,143


In [8]:
def matched_inference(
    matched,
    seed=607,
    n_permutations=9_999,
    n_bootstrap=5_000,
):
    """
    Run matched inference using:
    1. One-sided sign test
    2. Within-set permutation test
    3. Matched-set percentile bootstrap CI
    """

    rng = np.random.default_rng(seed)

    # Listed outcome in each matched set
    listed_values = (
        matched[
            matched["role"] == "listed"
        ]
        .set_index("set_id")
        ["y"]
    )

    # Mean private outcome in each matched set
    private_values = (
        matched[
            matched["role"] == "private"
        ]
        .groupby("set_id")
        ["y"]
        .mean()
    )

    # Matched gap
    differences = (
        listed_values
        - private_values
    ).to_numpy()

    # ------------------------------------------------
    # 1. One-sided sign test
    # ------------------------------------------------

    nonzero_differences = differences[
        differences != 0
    ]

    number_positive = int(
        (nonzero_differences > 0).sum()
    )

    sign_test = stats.binomtest(
        number_positive,
        len(nonzero_differences),
        p=0.5,
        alternative="greater",
    )

    # ------------------------------------------------
    # 2. Observed median gap
    # ------------------------------------------------

    observed_median = np.median(
        differences
    )

    # Store the outcomes within each matched set.
    # The first value is listed; the remaining values
    # are its matched private controls.
    values_by_set = []

    for _, group in matched.groupby("set_id"):

        listed_y = group.loc[
            group["role"] == "listed",
            "y",
        ]

        private_y = group.loc[
            group["role"] == "private",
            "y",
        ]

        set_values = pd.concat(
            [listed_y, private_y]
        ).to_numpy()

        values_by_set.append(set_values)

    # ------------------------------------------------
    # 3. Within-set permutation test
    # ------------------------------------------------

    permutation_medians = np.empty(
        n_permutations
    )

    for b in range(n_permutations):

        permutation_gaps = []

        for values in values_by_set:

            # Randomly assign one firm in the set
            # to receive the "listed" label
            selected_index = rng.integers(
                0,
                len(values),
            )

            permuted_listed = values[
                selected_index
            ]

            permuted_private_mean = np.delete(
                values,
                selected_index,
            ).mean()

            permutation_gaps.append(
                permuted_listed
                - permuted_private_mean
            )

        permutation_medians[b] = np.median(
            permutation_gaps
        )

    permutation_p = (
        1
        + np.sum(
            permutation_medians
            >= observed_median
        )
    ) / (
        n_permutations + 1
    )

    # ------------------------------------------------
    # 4. Bootstrap matched sets
    # ------------------------------------------------

    bootstrap_indices = rng.integers(
        0,
        len(differences),
        size=(
            n_bootstrap,
            len(differences),
        ),
    )

    bootstrap_medians = np.median(
        differences[bootstrap_indices],
        axis=1,
    )

    ci_low_log, ci_high_log = np.percentile(
        bootstrap_medians,
        [2.5, 97.5],
    )

    # ------------------------------------------------
    # Return results
    # ------------------------------------------------

    return {
        "Matched sets": len(differences),
        "Listed larger": number_positive,
        "Share listed larger": (
            number_positive
            / len(nonzero_differences)
        ),
        "Median log gap": observed_median,
        "Median ratio": np.exp(
            observed_median
        ),
        "CI low": np.exp(ci_low_log),
        "CI high": np.exp(ci_high_log),
        "Sign-test p-value": sign_test.pvalue,
        "Permutation p-value": permutation_p,
    }

In [9]:
results_main = matched_inference(
    matched_assets,
    seed=607,
)

results_revenue = matched_inference(
    matched_revenue,
    seed=607,
)

results_trimmed = matched_inference(
    matched_trimmed,
    seed=607,
)

In [10]:
robustness_results = pd.DataFrame(
    [
        {
            "Specification": "Main assets",
            **results_main,
        },
        {
            "Specification": (
                "Operating revenue"
            ),
            **results_revenue,
        },
        {
            "Specification": (
                "Assets excluding top 1%"
            ),
            **results_trimmed,
        },
    ]
)

robustness_results

,Specification,Matched sets,Listed larger,Share listed larger,Median log gap,Median ratio,CI low,CI high,Sign-test p-value,Permutation p-value
0,Main assets,351,300,0.854701,2.074373,7.959552,6.066219,11.111425,2.396202e-44,0.0001
1,Operating revenue,346,259,0.748555,1.707239,5.513716,3.923796,7.461218,2.818514e-21,0.0001
2,Assets excluding top 1%,348,306,0.879310,2.225097,9.254379,6.786480,13.146728,6.079271e-51,0.0001


In [11]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

robustness_results.to_csv(
    OUTPUT_FILE,
    index=False,
)

print("Saved to:", OUTPUT_FILE)

Saved to: /Users/jiangminglan/ds607-singapore-firm-size/outputs/tables/robustness_outcomes.csv


The main matched analysis estimates that listed firms are approximately 7.96 times larger than comparable private firms in terms of total assets. When operating revenue is used as an alternative measure of firm size, the estimated median ratio remains substantial at approximately 5.51, with a 95% bootstrap confidence interval of 3.92 to 7.46. After excluding the largest 1% of firms separately from the listed and private samples, the estimated median asset ratio is approximately 9.25, with a 95% confidence interval of 6.79 to 13.15. All three within-set permutation tests produce p-values below 0.001. Therefore, the main conclusion is robust to the alternative size measure and is not driven solely by the largest firms.